> This post stakes a position: agent memory is an architecture problem with an architecture solution.

# Introduction

A typical criticism of agents is that they "forget everything between sessions."

And as it goes with these arguments, the word "forget" is load-bearing. It implies a retention mechanism that failed. But agents don't forget - there's no architecture for remembering. Each session starts blank not because memories faded, but because no one built the filing system.

`helix` is that filing system. This post is my contribution to the discourse on agent memory - specifically, the claim that *feedback* is what separates storage from learning. `ftl` persisted artifacts. `helix` tracks which artifacts actually helped. The difference compounds.

I've been building with Opus 4.5 since its release. The model capability is there - Opus 4.5 proved agents can be genuine collaborators. What was missing was architecture that let collaboration compound across sessions. `helix` provides that architecture.

# Philosophy

Six principles, each diagnosing a specific failure mode:

| Principle | The Failure It Prevents |
|-----------|------------------------|
| **Feedback closes the loop** | Memory accumulation without learning |
| **Verify first** | Work that can't prove it succeeded |
| **Bounded scope** | Unauditable agent modifications |
| **Present over future** | Premature abstraction, over-engineering |
| **Edit over create** | File proliferation, duplicate logic |
| **Blocking is success** | Token spiral on unsolvable problems |

The word "feedback" in the first principle names the mechanism most agent memory systems lack. `ftl` persisted artifacts - each task left traces that survived. But persistence is not learning. A filing cabinet that grows larger is not getting smarter.

`helix` closes the loop. When memory is injected into a task, we track whether it helped. What happens when a memory is injected but ignored? The system notices. That memory's effectiveness score drops, pushing it down in future rankings. Memories that consistently help rise. Memories that seemed relevant but weren't sink.

I believe this is the critical difference between storage and learning: tracking not what exists, but what worked. Most agent memory systems fail here - they optimize retrieval relevance when they should optimize retrieval *usefulness*.

# The Development Loop

```
/helix <objective>
    │
    ▼
┌─────────────────────────────────────┐
│  EXPLORER (haiku, 6 tools)          │
│  structure │ patterns │ memory │ targets
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  PLANNER (opus)                     │
│  Decompose → Dependencies → Budget  │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  BUILDER (opus, budget 5-9)         │
│  Read → Implement → Verify → Report │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  OBSERVER (opus)                    │
│  Extract failures │ Chunk patterns  │
└─────────────────────────────────────┘
```

Four agents. Explorer gathers context. Planner decomposes objectives into task DAGs. Builder executes within strict constraints. Observer extracts learning from outcomes.

Memory flows through the entire pipeline:

```
recall() → inject → feedback() → store()
```

The Explorer queries memory before planning. The Builder receives injected memories and reports which ones helped. The Observer stores new failures and patterns. The feedback loop adjusts rankings.

What happens when the Builder reports that a memory was never used? The feedback loop records a failure. That memory's effectiveness score drops. Next time, it ranks lower - or doesn't surface at all. The system learns which context actually helps, not which context *seemed* relevant at query time.

Each completed objective makes the system smarter. But only if feedback is honest - and the architecture enforces honesty by comparing what was injected against what was utilized.

# Agents

At one extreme: a single monolithic agent doing everything. Maximum context, minimum coordination overhead. But also maximum scope creep, maximum token waste when things go sideways.

At the other extreme: dozens of specialized microservices, each responsible for one atomic operation. Minimal blast radius, but coordination overhead dominates. The orchestrator becomes more complex than the work.

`helix` sits between: four agents, distinct capabilities, explicit constraints.

| Agent | Model | Role | Budget |
|-------|-------|------|--------|
| **Explorer** | Haiku | Codebase reconnaissance | 6 |
| **Planner** | Opus | Task DAG decomposition | unlimited |
| **Builder** | Opus | Execution within constraints | 5-9 |
| **Observer** | Opus | Learning extraction | 10 |

The division is deliberate.

**Explorer** runs on Haiku for cost efficiency. Its job is reconnaissance, not reasoning - find the structure, detect the framework, query memory, identify targets. Six tool calls. More would be scope creep.

**Planner** runs on Opus with no budget limit. Planning is highest-leverage - a poor plan wastes every downstream tool call. The Planner decomposes complex objectives into focused tasks, sets dependencies, assigns budgets.

**Builder** runs on Opus with tight budget (5-9 tools per task). This constraint prevents spiral. Can't complete within budget? Block with what was tried. The budget forces focus: read delta files, implement, verify, report.

**Observer** runs on Opus with enough budget to analyze outcomes. Its job: extract generalizable knowledge from what just happened. Failures from blocked tasks. Patterns from successful completions. Relationships between memories.

# Task DAG

The Planner doesn't create a list. It creates a directed acyclic graph with explicit dependencies.

```
001: spec-auth-models ─┬─→ 002: spec-auth-tests ─┬─→ 004: impl-auth-routes
                       │                         │
                       └─→ 003: impl-auth-service ─┘
```

Each task has exactly these fields:

| Field | Purpose |
|-------|--------|
| `seq` | Execution order ("001", "002") |
| `slug` | Human-readable name ("spec-auth-models") |
| `objective` | What this task accomplishes |
| `delta` | Files this task may modify (strict constraint) |
| `verify` | Command to verify completion |
| `depends` | Tasks that must complete first |
| `budget` | Tool calls allocated (5-9) |

Tasks register with Claude Code's native task system, visible via `Ctrl+T`. Humans see progress as the pipeline executes.

What happens when task 002 fails? The DAG structure means 003 can proceed if it doesn't depend on 002. Failures are contained to their branch. A blocked task doesn't poison the entire objective - only tasks that explicitly depend on it.

`002` and `003` can execute simultaneously once `001` completes. The DAG captures both sequencing constraints and parallelization opportunities.

# Memory System

This is where `helix` diverges most from `ftl`. Not storage - a learning system with effectiveness tracking and decay.

## Storage

Everything lives in a single SQLite database at `.helix/helix.db`. No scattered JSON files, no complex hierarchies.

| Table | Purpose |
|-------|--------|
| `memory` | Failures and patterns with embeddings |
| `memory_edge` | Relationships between memories |
| `exploration` | Gathered context from Explorer |
| `plan` | Task decompositions from Planner |
| `workspace` | Task execution contexts |

Memories are stored with 384-dimensional embeddings for semantic search. 384 dimensions. Our brains did not evolve to visualize a 384-dimensional space any more than they evolved to visualize the distance to Andromeda. But similarity in that space captures meaning in ways keyword matching never will. Two memories about "authentication failing silently" and "auth errors swallowed without logging" cluster together - zero shared keywords.

## Scoring Formula

Ranking is not just semantic relevance:

```
score = (0.5 × relevance) + (0.3 × effectiveness) + (0.2 × recency)
```

Where:
- **relevance** = cosine similarity between query and memory embeddings
- **effectiveness** = `helped / (helped + failed)`, default 0.5 if no feedback
- **recency** = `2^(-days_since_use / 7)` (ACT-R decay)

The ACT-R decay comes from cognitive architecture research. Memories that haven't been used recently fade, matching how human memory works. A memory that helped six months ago but hasn't been touched since ranks lower than one used yesterday.

## The Feedback Loop

The critical function:

```python
feedback(utilized, injected)
# utilized memories: helped++
# injected-but-unused: failed++
```

When the Builder completes a task, it reports which memories were utilized. The feedback function compares this to what was injected:

- Memory injected AND utilized → `helped` increments
- Memory injected but NOT utilized → `failed` increments

Over time, effective memories rise. Memories that consistently get injected but ignored sink.

I believe this is why most agent memory systems fail: they track what was stored, not what helped. Retrieval without feedback is search. Retrieval with feedback is learning. The distinction matters - I explored this more in my post on `ftl`, and the failure mode is worth naming: *relevance optimization* when the system should optimize *usefulness*.

## SOAR Chunking

The Observer extracts patterns using SOAR-style chunking. When a task succeeds with a notable technique:

```bash
python3 $HELIX_PLUGIN_ROOT/lib/memory/core.py chunk \
    --task "What was accomplished" \
    --outcome "SUCCESS" \
    --approach "The technique that worked"
```

This captures the transition from deliberate problem-solving to compiled expertise. A technique that worked once becomes a retrievable pattern for similar situations.

## Graph Relationships

Memories don't exist in isolation. The system tracks edges:

| Type | Meaning |
|------|--------|
| `co_occurs` | These failures tend to appear together |
| `causes` | This failure leads to that failure |
| `solves` | This pattern resolves that failure |
| `similar` | These memories are semantically close |

What happens when a new failure is stored? The `connected()` traversal explores its neighborhood - if this failure appears, what related failures should we watch for? What patterns have solved it before? The graph surfaces context that pure embedding similarity misses.

## Maintenance

Three operations:

| Operation | What it does |
|-----------|-------------|
| `consolidate()` | Merge semantically similar memories |
| `prune()` | Remove memories with effectiveness < 0.25 |
| `decay()` | Find dormant memories that haven't been used |

# Commands

| Command | Purpose |
|---------|--------|
| `/helix <objective>` | Full pipeline: explore → plan → build → observe |
| `/helix-query "topic"` | Search memory by semantic similarity |
| `/helix-stats` | Memory health metrics and feedback loop status |

# Blocking as Learning

The instinct is to keep trying. An agent that gives up feels like failure.

The word "failure" names the wrong thing here. A blocked workspace with clear documentation is *information*. An agent that spirals for 100k tokens is *waste*. The confidence to escalate - "this is beyond what I can debug, here's what I tried" - is a feature.

When a task goes sideways, the Builder has a hard constraint: 5-9 tools max. If it hasn't solved the problem within budget, it's exploring, not debugging. At that point, or after hitting the same error three times, the Builder blocks.

The workspace records what was tried:

```
BLOCKED: Need to modify src/main.py but it's not in delta
TRIED: Implemented auth service in src/services/auth.py
ERROR: Cannot import auth routes without modifying main.py
```

The metacognition check is explicit: after three failed attempts with similar approaches, stop and analyze. Is there a fundamentally different approach? Is the task mis-scoped? Is information missing?

Blocked workspaces feed the Observer. Every block is a potential failure pattern to extract - a lesson for future tasks. The system learns from failures, but only if failures are captured cleanly instead of buried in spiral.

Put frankly: a blocked workspace with good documentation is a contribution. Token spiral is not.

# When to Use

**Use helix when:**

- Work should persist as learned knowledge that compounds
- Complex objectives need decomposition into verifiable tasks
- Bounded, reviewable scope with explicit file constraints matters
- Past failures should inform future attempts
- Framework-specific development benefits from detected idioms

**Skip helix when:**

- Simple single-file changes don't benefit from orchestration
- Exploratory prototyping where the model should wander
- Quick one-offs with no future value

I reach for `helix` when the work will matter again tomorrow. One-off script debugging doesn't need orchestration overhead. Infrastructure that will evolve over months - the memory system pays dividends every session.

Knowing when to reach for these tools - and when not to - is itself a skill worth developing.

# Installation

```bash
# Add the crinzo-plugins marketplace
claude plugin marketplace add https://github.com/enzokro/crinzo-plugins

# Install helix
claude plugin install helix@crinzo-plugins
```

Or from inside Claude Code:

```bash
/plugin marketplace add https://github.com/enzokro/crinzo-plugins
/plugin install helix@crinzo-plugins
```

# Conclusion

Context loss is an architecture problem, not a capability problem. The models are ready - Opus 4.5 proved agents can be genuine collaborators. What was missing was architecture that let collaboration compound.

`helix` builds on `ftl` with a crucial addition: feedback. Memory accumulation is not learning. A system that tracks which memories actually helped - and adjusts rankings based on that signal - is a system that improves.

In short: memory without feedback is storage. Memory with feedback is learning.

The models are ready. The architecture is learning. We're currently closer to "useful tool" than "genuine collaborator" on the spectrum I charted in my agents post - but the gap is closing. The question is what we'll build when context compounds instead of vanishing.